In [31]:
import torch
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [32]:
train_data = datasets.CIFAR10(root='data', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.CIFAR10(root='data', train=False, download=True, transform=transforms.ToTensor())

In [33]:
train_data[0][0].shape

torch.Size([3, 32, 32])

In [34]:
len(train_data.classes)

10

In [35]:
class AlexNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.feature = nn.Sequential(
        nn.Conv2d(3,16,kernel_size=3,stride=1,padding=1),
        nn.MaxPool2d(kernel_size=2,stride=2),
        nn.ReLU(),
        nn.Conv2d(16,32,kernel_size=3,stride=1,padding=1),
        nn.MaxPool2d(kernel_size=2,stride=2),
        nn.ReLU(),
        nn.Conv2d(32,64,kernel_size=3,stride=1,padding=1),
        #nn.MaxPool2d(kernel_size=2,stride=2),
        nn.ReLU(),
        nn.Conv2d(64,128,kernel_size=3,stride=1,padding=1),
        #nn.MaxPool2d(kernel_size=2,stride=2),
        nn.ReLU(),
        nn.Conv2d(128,256,kernel_size=3,stride=1,padding=1),
        nn.MaxPool2d(kernel_size=2,stride=2),
        nn.ReLU()

    )


    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(256*4*4,1000),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(1000,516),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(516,128),
        nn.ReLU(),
        nn.Linear(128,10)
    )

  def forward(self,x):

    output = self.feature(x)
    output = self.classifier(output)
    return output



In [36]:
dummy = torch.randn(1, 3, 32, 32)
model = AlexNet()
print(model.feature(dummy).shape)

torch.Size([1, 256, 4, 4])


In [37]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)  # confirm GPU is active

cuda


In [38]:
model = AlexNet().to(device)

In [39]:
lr=0.001
epochs = 30
loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr)

In [40]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

In [41]:
for epoch in range(epochs):
  for images,labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)
    optimizer.zero_grad()
    output = model(images)
    loss_value=loss(output,labels)
    loss_value.backward()
    optimizer.step()
  print(f'Epoch: {epoch + 1}, Loss: {loss_value.item()}')

Epoch: 1, Loss: 1.2886278629302979
Epoch: 2, Loss: 0.9268962144851685
Epoch: 3, Loss: 0.8827494382858276
Epoch: 4, Loss: 1.4683839082717896
Epoch: 5, Loss: 1.4967334270477295
Epoch: 6, Loss: 0.7714874744415283
Epoch: 7, Loss: 0.6770466566085815
Epoch: 8, Loss: 0.3968227505683899
Epoch: 9, Loss: 0.7994441390037537
Epoch: 10, Loss: 0.40382498502731323
Epoch: 11, Loss: 0.29370880126953125
Epoch: 12, Loss: 0.8827267289161682
Epoch: 13, Loss: 0.3129173517227173
Epoch: 14, Loss: 0.35939154028892517
Epoch: 15, Loss: 0.5635889768600464
Epoch: 16, Loss: 0.07666654139757156
Epoch: 17, Loss: 0.5073938965797424
Epoch: 18, Loss: 0.39096853137016296
Epoch: 19, Loss: 0.154677152633667
Epoch: 20, Loss: 0.17546549439430237
Epoch: 21, Loss: 0.1640104055404663
Epoch: 22, Loss: 0.14039567112922668
Epoch: 23, Loss: 0.19343391060829163
Epoch: 24, Loss: 0.22900980710983276
Epoch: 25, Loss: 0.5605143308639526
Epoch: 26, Loss: 0.4473901391029358
Epoch: 27, Loss: 0.34978893399238586
Epoch: 28, Loss: 0.501812756

In [44]:
test_loader = DataLoader(test_data, batch_size=64, shuffle=True)

In [45]:
correct=0
total=0
for images,labels in test_loader:
    images = images.to(device)
    labels = labels.to(device)
    output = model(images)
    predicted = torch.argmax(output, dim=1)
    total +=labels.size(0)
    correct += (predicted==labels).sum().item()
    accuracy = 100*correct/total
print(f'Accuracy: {accuracy:.2f}%')



Accuracy: 71.32%
